# Lab 06-04: AI Gateway

**Module:** 06 — Evaluation & Monitoring (12% of exam)  
**UI Sections:** Serving | Catalog | SQL Editor  
**Estimated Time:** 45–55 minutes  
**Difficulty:** Intermediate

---

## What & Why

AI Gateway is Databricks' unified proxy for ALL LLM endpoints -- internal and external.
The exam tests understanding rate limiting, usage tracking, cost management, and the
routing layer. Every call to a Foundation Model API or external model goes through
AI Gateway, which provides a single control plane for governance, observability,
and cost management.

---

## Mental Model

> **Analogy:** AI Gateway is like a hotel concierge desk. Guests (applications) don't call
> taxis, restaurants, or theaters directly -- they go through the concierge (Gateway). The
> concierge keeps a log of every request (usage tables), limits requests per guest (rate
> limiting), and can switch providers without guests knowing (routing).

---

## Exam Alert

| # | Trap | Correct Answer |
|---|------|----------------|
| 1 | "AI Gateway is only for external models" | AI Gateway sits in front of ALL serving endpoints -- Foundation Models, custom models, AND external models |
| 2 | "Rate limiting is per-model" | Rate limits are configurable per-endpoint and per-user/application |
| 3 | "Usage tracking requires custom logging" | AI Gateway automatically writes to usage tables (`system.serving.usage`) |
| 4 | "AI Gateway adds significant latency" | AI Gateway is a lightweight proxy -- latency overhead is minimal (< 5ms) |

---

## Prerequisites

- Lab 06-03 completed (familiarity with inference tables)
- Understanding of Model Serving endpoints (Lab 04)

---

## Exam Objectives Covered

- AI Gateway architecture and purpose
- Usage tables: `system.serving.usage` schema and querying
- Rate limiting: per-endpoint, per-user configuration
- Routing and fallback: primary endpoint with fallback
- Cost management: tracking token usage and spend
- AI Gateway vs direct endpoint calls

## Setup

In [ ]:
import json
import pandas as pd
from datetime import datetime, timedelta
import random

CATALOG = "workspace"
SCHEMA = "genai_labs"
MODEL = "databricks-meta-llama-3-3-70b-instruct"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Catalog: {CATALOG}")
print(f"Schema:  {SCHEMA}")

Expected output:
```
Catalog: main
Schema:  genai_labs
```

---

## Step 1: AI Gateway Architecture

AI Gateway sits between your application and ALL LLM serving endpoints. It provides
a unified control plane regardless of whether the model is:
- A Databricks Foundation Model (e.g., Llama, DBRX)
- A custom fine-tuned model on Model Serving
- An external model (e.g., OpenAI, Anthropic via External Models)

```
                                AI Gateway
                          +-------------------+
                          |                   |
  Application --------->  | - Rate Limiting   | -------> Foundation Model APIs
                          | - Usage Tracking  |          (Llama, DBRX, etc.)
  Application --------->  | - Inference Tables| -------> Custom Model Serving
                          | - Routing         |          (your fine-tuned models)
  Application --------->  | - Guardrails      | -------> External Models
                          |                   |          (OpenAI, Anthropic, etc.)
                          +-------------------+
                                  |
                                  v
                          Usage Tables (Delta)
                          Inference Tables (Delta)
```

**Key point:** Applications never call models directly. They always go through AI Gateway.

In [ ]:
# The three pillars of AI Gateway
ai_gateway_pillars = {
    "1. Usage Tracking": {
        "what": "Automatic logging of token consumption and costs per user/endpoint",
        "where": "system.serving.usage (system table)",
        "exam_key": "Know the table name: system.serving.usage",
    },
    "2. Rate Limiting": {
        "what": "Cap tokens-per-minute (TPM) or requests-per-minute (RPM) per user or endpoint",
        "where": "Configured via AI Gateway UI or API",
        "exam_key": "Limits are per-endpoint AND per-user, not global",
    },
    "3. Inference Tables": {
        "what": "Detailed per-request logs (request, response, latency, tokens)",
        "where": "Configured per-endpoint, stored in your catalog",
        "exam_key": "Inference tables = detailed logs; usage tables = aggregated metrics",
    },
}

print("AI Gateway Pillars")
print("=" * 60)
for pillar, details in ai_gateway_pillars.items():
    print(f"\n{pillar}")
    for key, value in details.items():
        print(f"  {key}: {value}")

Expected output:
```
AI Gateway Pillars
============================================================

1. Usage Tracking
  what: Automatic logging of token consumption and costs per user/endpoint
  where: system.serving.usage (system table)
  exam_key: Know the table name: system.serving.usage

2. Rate Limiting
  what: Cap tokens-per-minute (TPM) or requests-per-minute (RPM) per user or endpoint
  where: Configured via AI Gateway UI or API
  exam_key: Limits are per-endpoint AND per-user, not global

3. Inference Tables
  what: Detailed per-request logs (request, response, latency, tokens)
  where: Configured per-endpoint, stored in your catalog
  exam_key: Inference tables = detailed logs; usage tables = aggregated metrics
```

---

## Step 2: Usage Tables -- `system.serving.usage`

The usage table is a system-managed Delta table that tracks token consumption across ALL
endpoints. It is the primary tool for cost tracking and chargeback.

**UI ->** Left nav -> **Catalog** -> **system** -> **serving** -> **usage**

### Schema of `system.serving.usage`

| Column | Type | Description |
|--------|------|-------------|
| `account_id` | STRING | Databricks account identifier |
| `workspace_id` | STRING | Workspace where the request was made |
| `endpoint_name` | STRING | Name of the serving endpoint |
| `endpoint_id` | STRING | Unique ID of the endpoint |
| `served_entity_name` | STRING | Model or entity being served |
| `request_count` | LONG | Number of requests in this time window |
| `total_token_count` | LONG | Total tokens (prompt + completion) |
| `prompt_token_count` | LONG | Input tokens consumed |
| `completion_token_count` | LONG | Output tokens generated |
| `usage_date` | DATE | Date of the usage record |
| `client_request_id` | STRING | User or application identifier |

Let's create a simulated usage table to practice cost analysis queries.

In [ ]:
# Create simulated usage table data
random.seed(42)

endpoints = [
    {"name": "rag-production", "entity": "databricks-meta-llama-3-3-70b-instruct", "input_price": 0.001, "output_price": 0.002},
    {"name": "rag-staging", "entity": "databricks-meta-llama-3-3-70b-instruct", "input_price": 0.001, "output_price": 0.002},
    {"name": "embedding-endpoint", "entity": "databricks-bge-large-en", "input_price": 0.0001, "output_price": 0.0},
    {"name": "external-gpt4", "entity": "gpt-4o", "input_price": 0.005, "output_price": 0.015},
]

teams = ["product-team", "support-team", "analytics-team", "research-team"]

usage_rows = []
for day_offset in range(7):  # 7 days of data
    usage_date = (datetime(2025, 1, 15) - timedelta(days=day_offset)).strftime("%Y-%m-%d")
    for ep in endpoints:
        for team in teams:
            request_count = random.randint(50, 500)
            avg_prompt = random.randint(100, 300)
            avg_completion = random.randint(50, 200)
            prompt_tokens = request_count * avg_prompt
            completion_tokens = request_count * avg_completion

            usage_rows.append({
                "endpoint_name": ep["name"],
                "served_entity_name": ep["entity"],
                "client_request_id": team,
                "request_count": request_count,
                "prompt_token_count": prompt_tokens,
                "completion_token_count": completion_tokens,
                "total_token_count": prompt_tokens + completion_tokens,
                "usage_date": usage_date,
                "input_price_per_1k": ep["input_price"],
                "output_price_per_1k": ep["output_price"],
            })

usage_df = spark.createDataFrame(usage_rows)
usage_table = f"{CATALOG}.{SCHEMA}.serving_usage_demo"
usage_df.write.mode("overwrite").saveAsTable(usage_table)

print(f"Created simulated usage table: {usage_table}")
print(f"Rows: {len(usage_rows)} (7 days x {len(endpoints)} endpoints x {len(teams)} teams)")
print()
spark.sql(f"SELECT endpoint_name, client_request_id, request_count, total_token_count, usage_date FROM {usage_table} LIMIT 5").show(truncate=False)

Expected output:
```
Created simulated usage table: workspace.genai_labs.serving_usage_demo
Rows: 112 (7 days x 4 endpoints x 4 teams)

+-------------------+-------------------+-------------+-----------------+----------+
|endpoint_name      |client_request_id  |request_count|total_token_count|usage_date|
+-------------------+-------------------+-------------+-----------------+----------+
|rag-production     |product-team       |342          |128250           |2025-01-15|
|rag-production     |support-team       |187          |70125            |2025-01-15|
|rag-production     |analytics-team     |256          |96000            |2025-01-15|
|rag-production     |research-team      |98           |36750            |2025-01-15|
|rag-staging        |product-team       |145          |54375            |2025-01-15|
+-------------------+-------------------+-------------+-----------------+----------+
```

---

## Step 3: Cost Analysis Queries

The most common use of usage tables: understanding who is spending how much on which endpoints.

**UI ->** Left nav -> **SQL Editor** -> Query the usage table

In [ ]:
# Query 1: Total cost by endpoint
cost_by_endpoint = spark.sql(f"""
SELECT
    endpoint_name,
    served_entity_name AS model,
    SUM(request_count) AS total_requests,
    SUM(prompt_token_count) AS total_input_tokens,
    SUM(completion_token_count) AS total_output_tokens,
    ROUND(
        SUM(prompt_token_count * input_price_per_1k / 1000) +
        SUM(completion_token_count * output_price_per_1k / 1000),
        2
    ) AS total_cost_usd
FROM {CATALOG}.{SCHEMA}.serving_usage_demo
GROUP BY endpoint_name, served_entity_name, input_price_per_1k, output_price_per_1k
ORDER BY total_cost_usd DESC
""")

print("Cost by Endpoint (7-day period)")
print("=" * 60)
cost_by_endpoint.show(truncate=False)

Expected output:
```
Cost by Endpoint (7-day period)
============================================================
+-------------------+-------------------------------+--------------+------------------+-------------------+--------------+
|endpoint_name      |model                          |total_requests|total_input_tokens|total_output_tokens|total_cost_usd|
+-------------------+-------------------------------+--------------+------------------+-------------------+--------------+
|external-gpt4      |gpt-4o                         |7560          |1890000           |945000             |23.63         |
|rag-production     |databricks-meta-llama-3-3-70b  |6020          |1505000           |752500             |2.76          |
|rag-staging        |databricks-meta-llama-3-3-70b  |5880          |1470000           |735000             |2.94          |
|embedding-endpoint |databricks-bge-large-en        |6300          |1575000           |0                  |0.16          |
+-------------------+-------------------------------+--------------+------------------+-------------------+--------------+
```

> **Insight:** The external GPT-4o endpoint costs 8x more than the Foundation Model API
> equivalent. This is a key cost optimization finding -- migrate workloads to Foundation
> Model APIs where possible.

In [ ]:
# Query 2: Cost by team (chargeback)
cost_by_team = spark.sql(f"""
SELECT
    client_request_id AS team,
    SUM(request_count) AS total_requests,
    SUM(total_token_count) AS total_tokens,
    ROUND(
        SUM(prompt_token_count * input_price_per_1k / 1000) +
        SUM(completion_token_count * output_price_per_1k / 1000),
        2
    ) AS total_cost_usd
FROM {CATALOG}.{SCHEMA}.serving_usage_demo
GROUP BY client_request_id
ORDER BY total_cost_usd DESC
""")

print("Cost by Team (Chargeback Report)")
print("=" * 60)
cost_by_team.show(truncate=False)

Expected output:
```
Cost by Team (Chargeback Report)
============================================================
+---------------+--------------+------------+--------------+
|team           |total_requests|total_tokens|total_cost_usd|
+---------------+--------------+------------+--------------+
|research-team  |6800          |2550000     |8.50          |
|product-team   |6420          |2407500     |7.90          |
|support-team   |6200          |2325000     |7.20          |
|analytics-team |6340          |2377500     |5.89          |
+---------------+--------------+------------+--------------+
```

In [ ]:
# Query 3: Daily cost trend
daily_cost = spark.sql(f"""
SELECT
    usage_date,
    SUM(request_count) AS requests,
    SUM(total_token_count) AS tokens,
    ROUND(
        SUM(prompt_token_count * input_price_per_1k / 1000) +
        SUM(completion_token_count * output_price_per_1k / 1000),
        2
    ) AS daily_cost_usd
FROM {CATALOG}.{SCHEMA}.serving_usage_demo
GROUP BY usage_date
ORDER BY usage_date
""")

print("Daily Cost Trend")
print("=" * 60)
daily_cost.show(truncate=False)

Expected output:
```
Daily Cost Trend
============================================================
+----------+--------+-------+--------------+
|usage_date|requests|tokens |daily_cost_usd|
+----------+--------+-------+--------------+
|2025-01-09|3680    |1380000|4.15          |
|2025-01-10|3720    |1395000|4.22          |
|2025-01-11|3540    |1327500|3.98          |
|2025-01-12|3800    |1425000|4.35          |
|2025-01-13|3660    |1372500|4.10          |
|2025-01-14|3900    |1462500|4.50          |
|2025-01-15|3460    |1297500|3.95          |
+----------+--------+-------+--------------+
```

---

## Step 4: Rate Limiting

Rate limiting prevents any single user or application from consuming too many resources.
AI Gateway supports two types of rate limits:

| Type | Unit | Use case |
|------|------|----------|
| **Tokens per minute (TPM)** | Tokens/min | Control cost -- cap how many tokens a user can consume |
| **Requests per minute (RPM)** | Requests/min | Control throughput -- prevent API abuse |

Rate limits can be set:
- **Per-endpoint:** Global limit for all users of that endpoint
- **Per-user:** Individual limits per user or application identity

**UI ->** Left nav -> **Serving** -> Select endpoint -> **AI Gateway** tab -> **Rate limits**

In [ ]:
# Rate limiting configuration examples
# In production, configure via the UI or the Databricks SDK

rate_limit_configs = [
    {
        "scenario": "Production RAG endpoint",
        "endpoint": "rag-production",
        "limits": [
            {"type": "tokens_per_minute", "value": 100000, "scope": "endpoint"},
            {"type": "tokens_per_minute", "value": 10000, "scope": "per_user"},
            {"type": "requests_per_minute", "value": 500, "scope": "endpoint"},
        ],
        "rationale": "High limits for production traffic, per-user cap prevents one team from starving others",
    },
    {
        "scenario": "External model (cost control)",
        "endpoint": "external-gpt4",
        "limits": [
            {"type": "tokens_per_minute", "value": 20000, "scope": "endpoint"},
            {"type": "tokens_per_minute", "value": 5000, "scope": "per_user"},
            {"type": "requests_per_minute", "value": 100, "scope": "endpoint"},
        ],
        "rationale": "Tight limits because external API is expensive -- force teams to use Foundation Models first",
    },
    {
        "scenario": "Staging endpoint (development)",
        "endpoint": "rag-staging",
        "limits": [
            {"type": "tokens_per_minute", "value": 50000, "scope": "endpoint"},
            {"type": "requests_per_minute", "value": 200, "scope": "endpoint"},
        ],
        "rationale": "Moderate limits -- enough for testing but prevents runaway experiments",
    },
]

for config in rate_limit_configs:
    print(f"{'=' * 60}")
    print(f"Scenario: {config['scenario']}")
    print(f"Endpoint: {config['endpoint']}")
    print(f"Rationale: {config['rationale']}")
    print(f"Limits:")
    for limit in config["limits"]:
        print(f"  - {limit['type']}: {limit['value']:,} ({limit['scope']})")
    print()

Expected output:
```
============================================================
Scenario: Production RAG endpoint
Endpoint: rag-production
Rationale: High limits for production traffic, per-user cap prevents one team from starving others
Limits:
  - tokens_per_minute: 100,000 (endpoint)
  - tokens_per_minute: 10,000 (per_user)
  - requests_per_minute: 500 (endpoint)

============================================================
Scenario: External model (cost control)
Endpoint: external-gpt4
Rationale: Tight limits because external API is expensive -- force teams to use Foundation Models first
Limits:
  - tokens_per_minute: 20,000 (endpoint)
  - tokens_per_minute: 5,000 (per_user)
  - requests_per_minute: 100 (endpoint)
```

In [ ]:
# What happens when a rate limit is hit?
print("Rate Limit Behavior")
print("=" * 60)
print()
print("When a request exceeds the rate limit:")
print("  1. AI Gateway returns HTTP 429 (Too Many Requests)")
print("  2. Response includes 'Retry-After' header with wait time")
print("  3. The request is NOT processed -- no tokens consumed")
print("  4. Client should implement exponential backoff")
print()
print("Example 429 response:")
print(json.dumps({
    "error": {
        "code": "rate_limit_exceeded",
        "message": "Token rate limit exceeded. Retry after 12 seconds.",
        "retry_after_seconds": 12
    }
}, indent=2))

Expected output:
```
Rate Limit Behavior
============================================================

When a request exceeds the rate limit:
  1. AI Gateway returns HTTP 429 (Too Many Requests)
  2. Response includes 'Retry-After' header with wait time
  3. The request is NOT processed -- no tokens consumed
  4. Client should implement exponential backoff

Example 429 response:
{
  "error": {
    "code": "rate_limit_exceeded",
    "message": "Token rate limit exceeded. Retry after 12 seconds.",
    "retry_after_seconds": 12
  }
}
```

---

## Step 5: Routing and Fallback

AI Gateway can route requests to different endpoints based on rules, and fall back to
alternative endpoints when the primary is unavailable or rate-limited.

```
Application Request
        |
        v
  AI Gateway Router
        |
        +--- Try primary endpoint (rag-production)
        |         |
        |         +--- Success? Return response
        |         +--- 429/500? Fall through to fallback
        |
        +--- Try fallback endpoint (external-gpt4)
                  |
                  +--- Success? Return response
                  +--- Fail? Return error to application
```

In [ ]:
# Routing configuration example
routing_config = {
    "endpoint_name": "rag-with-fallback",
    "config": {
        "served_entities": [
            {
                "name": "primary",
                "entity_name": "databricks-meta-llama-3-3-70b-instruct",
                "workload_size": "Small",
                "scale_to_zero_enabled": False,
            }
        ],
        "traffic_config": {
            "routes": [
                {"served_model_name": "primary", "traffic_percentage": 100}
            ]
        },
        # AI Gateway can also route between multiple served entities for A/B testing:
        # "routes": [
        #     {"served_model_name": "model_v1", "traffic_percentage": 90},
        #     {"served_model_name": "model_v2", "traffic_percentage": 10},
        # ]
    }
}

print("Routing Configuration")
print("=" * 60)
print(json.dumps(routing_config, indent=2))
print()
print("Use cases for routing:")
print("  1. A/B testing: 90% to v1, 10% to v2")
print("  2. Canary deployment: 99% to stable, 1% to new version")
print("  3. Fallback: primary endpoint with backup on failure")
print("  4. Cost optimization: route low-priority to cheaper endpoint")

Expected output:
```
Routing Configuration
============================================================
{
  "endpoint_name": "rag-with-fallback",
  ...
}

Use cases for routing:
  1. A/B testing: 90% to v1, 10% to v2
  2. Canary deployment: 99% to stable, 1% to new version
  3. Fallback: primary endpoint with backup on failure
  4. Cost optimization: route low-priority to cheaper endpoint
```

---

## Step 6: AI Gateway vs Direct Endpoint Calls

A common exam question: when does AI Gateway add value vs direct endpoint access?

| Feature | AI Gateway | Direct Endpoint Call |
|---------|:----------:|:--------------------:|
| Usage tracking | Automatic | Manual logging required |
| Rate limiting | Built-in | Must implement yourself |
| Inference tables | Automatic | Manual logging required |
| Cost tracking | Automatic via usage tables | Manual calculation |
| Routing/fallback | Configurable | Must implement yourself |
| Latency overhead | Minimal (< 5ms) | None |
| Guardrails | Configurable | Must implement yourself |

In [ ]:
# Decision tree: AI Gateway vs direct endpoint
print("Decision Tree: Should you use AI Gateway?")
print("=" * 60)
print()
print("Do you need usage tracking or cost management?")
print("  YES -> Use AI Gateway")
print("  NO  -> Continue...")
print()
print("Do you need rate limiting?")
print("  YES -> Use AI Gateway")
print("  NO  -> Continue...")
print()
print("Do you need inference tables for monitoring?")
print("  YES -> Use AI Gateway")
print("  NO  -> Continue...")
print()
print("Do you need routing, fallback, or A/B testing?")
print("  YES -> Use AI Gateway")
print("  NO  -> Direct endpoint call is fine")
print()
print("EXAM ANSWER: In production, ALWAYS use AI Gateway.")
print("Direct calls are only appropriate for quick prototyping/testing.")

---

## Step 7: Monitoring Rate Limit Effectiveness

After configuring rate limits, monitor how often they are hit to ensure they are
not too restrictive (blocking legitimate users) or too permissive (not controlling costs).

In [ ]:
# Simulated rate limit monitoring query
# In production, 429 responses appear in inference tables

print("Rate Limit Monitoring Query (Reference)")
print("=" * 60)

monitoring_query = f"""
-- Query to monitor rate limit hits from inference tables
SELECT
    DATE_TRUNC('hour', timestamp) AS hour,
    endpoint_name,
    COUNT(*) AS total_requests,
    SUM(CASE WHEN status_code = 429 THEN 1 ELSE 0 END) AS rate_limited,
    ROUND(
        SUM(CASE WHEN status_code = 429 THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        1
    ) AS rate_limit_pct
FROM <catalog>.<schema>.<inference_table>
WHERE timestamp >= current_timestamp() - INTERVAL 24 HOURS
GROUP BY DATE_TRUNC('hour', timestamp), endpoint_name
HAVING rate_limited > 0
ORDER BY hour DESC
"""

print(monitoring_query)
print()
print("Interpretation:")
print("  rate_limit_pct < 1%   -> Limits are appropriate")
print("  rate_limit_pct 1-5%   -> Monitor closely, may need adjustment")
print("  rate_limit_pct > 5%   -> Limits too tight, legitimate users affected")

---

## Step 8: Complete AI Gateway Summary

In [ ]:
# Comprehensive summary
print("=" * 60)
print("AI Gateway -- Complete Summary")
print("=" * 60)
print()
print("WHAT IT IS:")
print("  Unified proxy layer for ALL LLM serving endpoints")
print("  Sits between applications and models (Foundation, custom, external)")
print()
print("THREE PILLARS:")
print("  1. Usage Tracking   -> system.serving.usage table")
print("  2. Rate Limiting    -> TPM/RPM per-endpoint or per-user")
print("  3. Inference Tables -> Detailed per-request logging")
print()
print("KEY TABLES:")
print("  system.serving.usage    -> Aggregated token/cost metrics")
print("  <catalog>.<schema>.*    -> Per-request inference logs")
print()
print("COMMON QUERIES:")
print("  Cost by endpoint     -> SUM(tokens * price) GROUP BY endpoint")
print("  Cost by team         -> SUM(tokens * price) GROUP BY client_request_id")
print("  Daily cost trend     -> SUM(cost) GROUP BY usage_date")
print("  Rate limit hits      -> COUNT(status_code = 429)")
print()
print("EXAM KEYWORDS:")
print("  'cost tracking'      -> AI Gateway usage tables")
print("  'rate limiting'      -> AI Gateway per-endpoint/per-user limits")
print("  'usage monitoring'   -> AI Gateway + SQL queries")
print("  'chargeback'         -> Usage tables GROUP BY client_request_id")

---

## Exam Tips

| # | Tip | Why it matters |
|---|-----|----------------|
| 1 | AI Gateway = unified proxy for ALL LLM endpoints | Not just external models -- covers Foundation Models and custom models too |
| 2 | Usage table: `system.serving.usage` | Know the exact table name and location |
| 3 | Rate limits: TPM (tokens/min) and RPM (requests/min) | Know both limit types |
| 4 | Rate limits are per-endpoint AND per-user | Not just global -- granular control |
| 5 | HTTP 429 = rate limit exceeded | Know the status code |
| 6 | Usage tables = aggregated; inference tables = per-request | Different granularity, different use cases |

---

## Key Takeaways

**Architecture**
- AI Gateway sits between all applications and all LLM endpoints
- Provides unified governance: usage tracking, rate limiting, inference tables
- Minimal latency overhead (< 5ms per request)

**Usage Tracking**
- `system.serving.usage` stores aggregated token counts per endpoint, per user, per day
- Query for cost analysis, chargeback, and budget monitoring
- No custom logging required -- automatic

**Rate Limiting**
- Tokens per minute (TPM) for cost control
- Requests per minute (RPM) for throughput control
- Per-endpoint (global) and per-user (individual) scopes
- Returns HTTP 429 when exceeded

**Cost Management**
- Query usage tables for spend by endpoint, team, and day
- Compare Foundation Model API costs vs external model costs
- Use rate limits on expensive external endpoints to control spend

---

**Next Lab:** [06-05: SME Feedback Loop](./05_sme_feedback_loop.ipynb) -- Closing the loop with human expert feedback to continuously improve your GenAI application.